<a href="https://colab.research.google.com/github/anosh1314/AMR_prediction/blob/main/AMR_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd

genes = pd.read_csv("/AccessoryGene.csv", index_col=0)    # strains x genes
meta = pd.read_csv("/Metadata.csv", index_col="Isolate")  # strains x antibiotics

drug = "CIP"                                   # ciprofloxacin
y = (meta[drug].dropna() == "R").astype(int)   # 1 = resistant, 0 = susceptible
X = genes.loc[y.index]                         # keep only strains that have an answer

# Drop genes that almost every strain has (or almost none): they can't help tell strains apart
freq = X.mean()
X = X.loc[:, (freq > 0.01) & (freq < 0.99)]

print("strains x genes:", X.shape)
print(y.value_counts())

strains x genes: (1935, 5993)
CIP
0    1506
1     429
Name: count, dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

models = {
    "Logistic regression": LogisticRegression(max_iter=2000, C=0.1, class_weight="balanced"),
    "Random forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    guesses = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    rows.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, guesses), 3),
        "Balanced accuracy": round(balanced_accuracy_score(y_test, guesses), 3),
        "ROC-AUC": round(roc_auc_score(y_test, probs), 3),
    })

results = pd.DataFrame(rows)
print(results)

                 Model  Accuracy  Balanced accuracy  ROC-AUC
0  Logistic regression     0.922              0.896    0.949
1        Random forest     0.943              0.893    0.968
2    Gradient boosting     0.941              0.900    0.972


In [10]:
rf = models["Random forest"]
top = pd.Series(rf.feature_importances_, index=X.columns).nlargest(10)
print(top)

nmpC           0.011699
pemI           0.010912
yjgH           0.008944
ybcQ           0.008656
group_984      0.008474
cat            0.007777
chpS           0.007567
group_8566     0.007564
group_11091    0.006986
mhpE           0.006714
dtype: float64
